In [0]:
dbutils.widgets.text("load_date", "")

In [0]:
from datetime import datetime

load_date = dbutils.widgets.get("load_date").strip()

if not load_date:
    raise ValueError(
        "load_date is required in YYYY-MM-DD format."
    )

try:
    datetime.strptime(load_date, "%Y-%m-%d")
except ValueError:
    raise ValueError(
        f"Invalid load_date={load_date}. "
        "Expected format: YYYY-MM-DD"
    )

print(f"Generating source data for load_date={load_date}")

In [0]:
catalog_name = "retail_lakehouse"
landing_schema = "landing"
volume_name = "source_files"

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS
{catalog_name}.{landing_schema}.{volume_name}
COMMENT 'Landing volume for synthetic retail source files'
""")

landing_base_path = (
    f"/Volumes/{catalog_name}/{landing_schema}/{volume_name}"
)

print(landing_base_path)

In [0]:
from datetime import date, datetime, timedelta
import random
import uuid

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

random.seed(42)

In [0]:
stores_data = [
    ("S001", "Bengaluru Central", "Bengaluru", "Karnataka", "South", "Large"),
    ("S002", "Mumbai Andheri", "Mumbai", "Maharashtra", "West", "Large"),
    ("S003", "Delhi Connaught Place", "Delhi", "Delhi", "North", "Large"),
    ("S004", "Chennai T Nagar", "Chennai", "Tamil Nadu", "South", "Medium"),
    ("S005", "Hyderabad Hitech City", "Hyderabad", "Telangana", "South", "Medium"),
    ("S006", "Pune Baner", "Pune", "Maharashtra", "West", "Medium"),
    ("S007", "Kolkata Salt Lake", "Kolkata", "West Bengal", "East", "Medium"),
    ("S008", "Jaipur Malviya Nagar", "Jaipur", "Rajasthan", "North", "Small"),
    ("S009", "Ahmedabad Satellite", "Ahmedabad", "Gujarat", "West", "Small"),
    ("S010", "Lucknow Gomti Nagar", "Lucknow", "Uttar Pradesh", "North", "Small")
]

stores_schema = [
    "store_id",
    "store_name",
    "city",
    "state",
    "region",
    "store_size"
]

stores_df = spark.createDataFrame(stores_data, stores_schema)

stores_df.show(truncate=False)

In [0]:
product_categories = {
    "Electronics": [
        ("P1001", "Wireless Headphones", "SoundMax", 2499.00, 1799.00),
        ("P1002", "Smart Watch", "FitPro", 4999.00, 3500.00),
        ("P1003", "Bluetooth Speaker", "SoundMax", 1999.00, 1300.00)
    ],
    "Home Appliances": [
        ("P2001", "Mixer Grinder", "HomeEase", 3499.00, 2500.00),
        ("P2002", "Electric Kettle", "HomeEase", 1499.00, 950.00),
        ("P2003", "Steam Iron", "QuickPress", 1899.00, 1200.00)
    ],
    "Fashion": [
        ("P3001", "Men Casual Shirt", "UrbanWear", 1299.00, 650.00),
        ("P3002", "Women Kurta", "EthnicLine", 1799.00, 900.00),
        ("P3003", "Running Shoes", "ActiveStep", 2999.00, 1800.00)
    ],
    "Grocery": [
        ("P4001", "Premium Rice 5kg", "DailyChoice", 749.00, 610.00),
        ("P4002", "Cooking Oil 5L", "HealthyDrop", 899.00, 760.00),
        ("P4003", "Dry Fruits Pack", "NutriBox", 1199.00, 850.00)
    ]
}

product_rows = []

for category, products in product_categories.items():
    for product_id, product_name, brand, selling_price, cost_price in products:
        product_rows.append(
            (
                product_id,
                product_name,
                category,
                brand,
                selling_price,
                cost_price,
                "ACTIVE"
            )
        )

products_schema = [
    "product_id",
    "product_name",
    "category",
    "brand",
    "standard_price",
    "cost_price",
    "product_status"
]

products_df = spark.createDataFrame(product_rows, products_schema)

products_df.show(truncate=False)

In [0]:
first_names = [
    "Aarav", "Vivaan", "Aditya", "Arjun", "Rohan",
    "Ananya", "Diya", "Ishita", "Meera", "Kavya"
]

last_names = [
    "Sharma", "Verma", "Gupta", "Patel", "Singh",
    "Mittal", "Rao", "Iyer", "Mehta", "Kapoor"
]

cities = [
    ("Bengaluru", "Karnataka", "South"),
    ("Mumbai", "Maharashtra", "West"),
    ("Delhi", "Delhi", "North"),
    ("Chennai", "Tamil Nadu", "South"),
    ("Hyderabad", "Telangana", "South"),
    ("Pune", "Maharashtra", "West"),
    ("Kolkata", "West Bengal", "East"),
    ("Jaipur", "Rajasthan", "North")
]

customer_rows = []

for customer_number in range(1, 501):
    first_name = random.choice(first_names)
    last_name = random.choice(last_names)
    city, state, region = random.choice(cities)

    age = random.randint(18, 70)
    registration_date = date.today() - timedelta(
        days=random.randint(30, 1500)
    )

    loyalty_tier = random.choices(
        ["Bronze", "Silver", "Gold", "Platinum"],
        weights=[45, 30, 20, 5],
        k=1
    )[0]

    customer_rows.append(
        (
            f"C{customer_number:05d}",
            f"{first_name} {last_name}",
            age,
            city,
            state,
            region,
            loyalty_tier,
            registration_date
        )
    )

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("region", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("registration_date", DateType(), True)
])

customers_df = spark.createDataFrame(
    customer_rows,
    schema=customers_schema
)

customers_df.show(10, truncate=False)

In [0]:
promotions_data = [
    (
        "PROMO001",
        "Electronics Weekend Sale",
        "Electronics",
        "PERCENTAGE",
        10.0,
        date.today() - timedelta(days=30),
        date.today() + timedelta(days=30)
    ),
    (
        "PROMO002",
        "Fashion Fest",
        "Fashion",
        "PERCENTAGE",
        15.0,
        date.today() - timedelta(days=15),
        date.today() + timedelta(days=15)
    ),
    (
        "PROMO003",
        "Grocery Saver",
        "Grocery",
        "FLAT",
        100.0,
        date.today() - timedelta(days=20),
        date.today() + timedelta(days=20)
    )
]

promotions_schema = StructType([
    StructField("promotion_id", StringType(), False),
    StructField("promotion_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("discount_type", StringType(), False),
    StructField("discount_value", DoubleType(), False),
    StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), False)
])

promotions_df = spark.createDataFrame(
    promotions_data,
    promotions_schema
)

promotions_df.show(truncate=False)

In [0]:
inventory_rows = []

for store in stores_data:
    store_id = store[0]

    for product in product_rows:
        product_id = product[0]

        opening_stock = random.randint(50, 300)
        received_quantity = random.randint(0, 100)
        sold_quantity = random.randint(0, opening_stock)
        damaged_quantity = random.randint(0, 5)

        closing_stock = (
            opening_stock
            + received_quantity
            - sold_quantity
            - damaged_quantity
        )

        inventory_rows.append(
            (
                store_id,
                product_id,
                date.today(),
                opening_stock,
                received_quantity,
                sold_quantity,
                damaged_quantity,
                max(closing_stock, 0)
            )
        )

inventory_schema = [
    "store_id",
    "product_id",
    "inventory_date",
    "opening_stock",
    "received_quantity",
    "sold_quantity",
    "damaged_quantity",
    "closing_stock"
]

inventory_df = spark.createDataFrame(
    inventory_rows,
    inventory_schema
)

inventory_df.show(10, truncate=False)

In [0]:
product_lookup = {
    product[0]: {
        "price": product[4],
        "category": product[2]
    }
    for product in product_rows
}

store_ids = [store[0] for store in stores_data]
customer_ids = [row[0] for row in customer_rows]
product_ids = list(product_lookup.keys())

status_codes = ["S", "C", "R"]
payment_methods = [
    "Credit Card",
    "Debit Card",
    "UPI",
    "Cash",
    "Wallet"
]

sales_channels = [
    "STORE",
    "ONLINE",
    "MOBILE_APP"
]

sales_rows = []

base_date = datetime.now() - timedelta(days=30)

for transaction_number in range(1, 10001):
    product_id = random.choice(product_ids)
    unit_price = product_lookup[product_id]["price"]

    quantity = random.randint(1, 5)
    discount_amount = round(
        random.choice([0, 0, 0, 50, 100, unit_price * 0.10]),
        2
    )

    transaction_timestamp = base_date + timedelta(
        seconds=random.randint(0, 30 * 24 * 60 * 60)
    )

    status_code = random.choices(
        status_codes,
        weights=[88, 5, 7],
        k=1
    )[0]

    sales_rows.append(
        (
            f"T{transaction_number:08d}",
            random.choice(customer_ids),
            product_id,
            random.choice(store_ids),
            transaction_timestamp.strftime("%Y-%m-%d %H:%M:%S"),
            str(quantity),
            str(unit_price),
            str(discount_amount),
            status_code,
            random.choice(payment_methods),
            random.choice(sales_channels),
            random.choice(["PROMO001", "PROMO002", "PROMO003", None])
        )
    )

In [0]:
for index in random.sample(range(len(sales_rows)), 50):
    row = list(sales_rows[index])
    row[1] = None
    sales_rows[index] = tuple(row)

In [0]:
for index in random.sample(range(len(sales_rows)), 30):
    row = list(sales_rows[index])
    row[5] = random.choice(["", "-1", "ABC", None])
    sales_rows[index] = tuple(row)

In [0]:
for index in random.sample(range(len(sales_rows)), 20):
    row = list(sales_rows[index])
    row[4] = random.choice([
        "INVALID_DATE",
        "31-02-2026",
        "",
        None
    ])
    sales_rows[index] = tuple(row)

In [0]:
for index in random.sample(range(len(sales_rows)), 25):
    row = list(sales_rows[index])
    row[2] = "P9999"
    sales_rows[index] = tuple(row)

In [0]:
for index in random.sample(range(len(sales_rows)), 15):
    row = list(sales_rows[index])
    row[8] = random.choice(["X", "UNKNOWN", ""])
    sales_rows[index] = tuple(row)

In [0]:
duplicate_rows = random.sample(sales_rows, 100)
sales_rows.extend(duplicate_rows)

In [0]:
len(sales_rows)

In [0]:
sales_schema = [
    "transaction_id",
    "customer_id",
    "product_id",
    "store_id",
    "transaction_timestamp",
    "quantity",
    "unit_price",
    "discount_amount",
    "status_code",
    "payment_method",
    "sales_channel",
    "promotion_id"
]

sales_df = spark.createDataFrame(
    sales_rows,
    sales_schema
)

sales_df.show(20, truncate=False)

In [0]:
print(f"Sales row count: {sales_df.count()}")

In [0]:
def write_csv_single_file(
    dataframe,
    output_path,
    file_name
):
    temporary_path = f"{output_path}/_temp"

    (
        dataframe
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .option("quoteAll", True)
        .csv(temporary_path)
    )

    files = dbutils.fs.ls(temporary_path)

    csv_file = [
        file.path
        for file in files
        if file.name.endswith(".csv")
    ][0]

    dbutils.fs.mkdirs(output_path)
    dbutils.fs.mv(
        csv_file,
        f"{output_path}/{file_name}"
    )

    dbutils.fs.rm(temporary_path, recurse=True)

    print(f"Created: {output_path}/{file_name}")

In [0]:
write_csv_single_file(
    sales_df,
    f"{landing_base_path}/sales/load_date={load_date}",
    f"sales_{load_date}.csv"
)

write_csv_single_file(
    products_df,
    f"{landing_base_path}/products",
    "products.csv"
)

write_csv_single_file(
    customers_df,
    f"{landing_base_path}/customers",
    "customers.csv"
)

write_csv_single_file(
    stores_df,
    f"{landing_base_path}/stores",
    "stores.csv"
)

write_csv_single_file(
    inventory_df,
    f"{landing_base_path}/inventory/load_date={load_date}",
    f"inventory_{load_date}.csv"
)

write_csv_single_file(
    promotions_df,
    f"{landing_base_path}/promotions",
    "promotions.csv"
)

In [0]:
display(
    dbutils.fs.ls(landing_base_path)
)

In [0]:
sales_source_path = (
    f"{landing_base_path}/sales/load_date={load_date}"
)

display(
    dbutils.fs.ls(sales_source_path)
)

In [0]:
validation_sales_df = (
    spark.read
    .option("header", True)
    .csv(sales_source_path)
)

validation_sales_df.show(10, truncate=False)

print(
    f"Rows read from CSV: {validation_sales_df.count()}"
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/retail_lakehouse/landing/source_files/sales"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/retail_lakehouse/landing/source_files/inventory"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/retail_lakehouse/landing/source_files/"
        "sales/load_date=2026-07-23"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/retail_lakehouse/landing/source_files/"
        "inventory/load_date=2026-07-23"
    )
)